# 1: XGBoost and SHAP for predicting use of thrombolysis

## Plain English summary
This notebook creates three important outputs. It first creates a model for predicting use of thrombolysis in SSNAP data. Then it creates a SHAP explainer that describes how the different features of the patient data influence the model's decision to thrombolyse. Lastly, it creates a list of the SHAP values from individual stroke teams and so ranks the stroke teams that are most likely to give thrombolysis.

This model is used in the thrombolysis decisions app to predict thrombolysis use.

![Flowchart of the process to select data and create the thrombolysis prediction model and the SHAP explainer model, and to use them to rank how likely hospitals are to give thrombolysis.](images/flowchart_xgboost_shap_benchmark_teams.png)

This model use 10 features:

* stroke_team_id
* stroke_severity
* prior_disability
* age
* infarction
* onset_to_arrival_time
* precise_onset_known
* onset_during_sleep
* arrival_to_scan_time
* afib_anticoagulant

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import pickle

from dataclasses import dataclass
from xgboost import XGBClassifier

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [2]:
import sys
sys.path.append('../../')  # path to dir containing stroke_utilities
import stroke_utilities.process_data as process_data
import stroke_utilities.models as models
import stroke_utilities.scenario as scenario

## Set up paths and filenames

In [3]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''

    data_read_path: str = '../../stroke_utilities/data/'
    output_folder = '../../stroke_utilities/output/'
    model_folder = '../../stroke_utilities/models'
    model_text = 'lgbm_decision_'
    notebook: str = '01_'

paths = Paths()

## Load data

Import the data from file:

In [4]:
# Load data
train = pd.read_csv(paths.data_read_path + 'cohort_10000_train.csv')
train['stroke_team_id'] = train['stroke_team_id'].astype('category')
test = pd.read_csv(paths.data_read_path + 'cohort_10000_test.csv')
test['stroke_team_id'] = test['stroke_team_id'].astype('category')

## Process data

Restrict the data to the following ten features, plus the "thrombolysis" feature.

In [5]:
# Put data into the format for model
features_to_model = [
    'stroke_team_id',
    'stroke_severity',
    'prior_disability',
    'age',
    'infarction',
    'onset_to_arrival_time',
    'precise_onset_known',
    'onset_during_sleep',
    'arrival_to_scan_time',
    'afib_anticoagulant',
    # 'year',    
    'thrombolysis'
]

In [6]:
train = train[features_to_model]
test = test[features_to_model]

In [7]:
train['stroke_team_id'].value_counts().sort_values()

stroke_team_id
85       87
95      106
109     198
102     209
63      280
       ... 
99     1544
71     1548
70     1579
6      1699
48     1756
Name: count, Length: 119, dtype: int64

If the year of admission to stroke team is still in the data, restrict it to the following range.

Currently this process does nothing because we have already removed "year" from the data.

In [8]:
# train = process_data.restrict_data_to_range(train, 2016, 2018, 'year')
# test = process_data.restrict_data_to_range(test, 2016, 2018, 'year')

Split the data. X contains the features for the model to use to predict use of thrombolysis, and y contains whether thrombolysis was used in the real data.

In [9]:
X_train, y_train = process_data.split_X_and_y(train, 'thrombolysis')
X_test, y_test = process_data.split_X_and_y(test, 'thrombolysis')

Check the list of features currently included in the X data:

In [10]:
features = list(X_train)

features

['stroke_team_id',
 'stroke_severity',
 'prior_disability',
 'age',
 'infarction',
 'onset_to_arrival_time',
 'precise_onset_known',
 'onset_during_sleep',
 'arrival_to_scan_time',
 'afib_anticoagulant']

For the XGBoost model, we need to change the single "stroke team ID" column to many individual team columns. For 119 separate teams, we will create 119 new columns. Each column may contain either 1 (meaning "yes") where a patient attended that stroke team, or 0 (meaning "no") where the patient did not attend that stroke team.

In [11]:
X_train = process_data.one_hot_encode_column(
    X_train, 'stroke_team_id', prefix='team')

X_test = process_data.one_hot_encode_column(
    X_test, 'stroke_team_id', prefix='team')

Check that the "stroke_team_id" column has gone and that there are now many "team_" columns.

In [12]:
# Get features
features_ohe = list(X_train)

# Print the first several...
print(features_ohe[:15])
# ... and last few feature names:
print(features_ohe[-3:])
# The remaining features are all "team_X" for increasing X.

['stroke_severity', 'prior_disability', 'age', 'infarction', 'onset_to_arrival_time', 'precise_onset_known', 'onset_during_sleep', 'arrival_to_scan_time', 'afib_anticoagulant', 'team_1', 'team_2', 'team_3', 'team_4', 'team_5', 'team_6']
['team_117', 'team_118', 'team_119']


In [13]:
team_cols = [c for c in X_train.columns if c.startswith('team_')]
X_train[team_cols] = X_train[team_cols].astype(int)
X_test[team_cols] = X_test[team_cols].astype(int)

## Fit model

Use an XGBoost model and teach it to predict the use of thrombolysis by using the training data set:

In [17]:
# Define and Fit model
model = XGBClassifier(verbosity = 0, seed=42, learning_rate=0.5)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.5, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

Save model to file:

In [18]:
# with open(f'{paths.model_folder}/model.p', 'wb') as fp:
#     pickle.dump(model, fp)
model.save_model(f'{paths.model_folder}/model_resave.json')

Load model back in:

In [18]:
model = XGBClassifier({'nthread': 4})  # init model
filename = f'{paths.model_folder}/model_resave.json'
model.load_model(filename)

Check how well the model works. Use the X_test test data set to predict the use of thrombolysis for patients that the model has never seen before, and see how closely the predictions match the true values in y_test.

In [19]:
# Get predictions
predicted = model.predict(X_test)
predicted_proba = model.predict_proba(X_test)[:,1]

# Show overall accuracy
accuracy = np.mean(predicted == y_test)
print (f'Accuracy: {accuracy:.3f}')

Accuracy: 0.827


## Create SHAP explainer

In units of log-odds:

In [19]:
explainer = shap.TreeExplainer(model)

[08:54:07] WARNING: /workspace/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.


Save SHAP model to file:

In [20]:
with open(f'{paths.model_folder}/shap_explainer_resave.p', 'wb') as fp:
    pickle.dump(explainer, fp)

In units of probability:

In [21]:
explainer_prob = shap.TreeExplainer(
    model, 
    data=X_train, 
    model_output="probability"
    )

[08:54:08] WARNING: /workspace/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.


Save SHAP model to file:

In [22]:
with open(f'{paths.model_folder}/shap_explainer_probability_resave.p', 'wb') as fp:
    pickle.dump(explainer_prob, fp)